## Part 3: Bragg Peak Masking

Because different corrections are needed when integrating Bragg peaks and diffuse data, it is important to ensure that the signals are separated before processing. It is important to note that a Bragg peaks' extent in $\phi$ can vary depending on the distance from the spindle axis in reciprocal space. And, Bragg peaks may not be perfectly symmetric Gaussian peaks. Peak width depends on crystal mosaicity, and this can even vary during a scan. In addition, the diffraction geometry may not be modeled exactly right, leading to further errors in peak position. A masking function must account for all of these effects. 

Designing a the perfect Bragg peak mask is a tricky problem, and different researchers have approached it various ways. With _mdx2_ we take a simple empirical approach. Here is the algorithm:

- List all of the pixels in the image stack above a certain count threshold ("hot pixels").
- Index all of the hot pixels, converting their position in image space to Miller index (_h_, _k_, _l_).
- Subtract the nearest integer from each Miller index leaving only the fractional part (_Δh_, _Δk_, _Δl_) which varies from -0.5 to 0.5. Because peaks have finite extent, the values of _Δh_, _Δk_, _Δl_ will not all be zero, but will look like a cloud of points.
- Fit an anisotropic 3D Gaussian distribution to the cloud of points.
- Flag outliers beyond a certain sigma threshold ("bad pixels")
- Create a mask for all pixels in the image stack excluding outliers and pixels falling within the 3D Gaussian region according to a sigma threshold.

To execute the masking altorithm in _mdx2_, first perform the strong pixel search:

In [1]:
!mdx2.find_peaks geometry.nxs data.nxs --count_threshold 20

Reading miller_index from geometry.nxs
  importing as MillerIndex from mdx2.geometry
Reading image_series from data.nxs
patching virtual field: /entry/image_series/data
  importing as ImageSeries from mdx2.data
finding pixels with counts above threshold: 20.0
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed:    1.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.4s
[Parallel(n_jobs=1)]: Done  24 tasks      | elapsed:    2.0s
[Parallel(n_jobs=1)]: Done  31 tasks      | elapsed:    2.5s
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    3.3s
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    4.0s
[Parallel(n_jobs=1)]: Done  60 tasks      | elapsed:    4.9s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    5.7s
[Parallel(n_jobs=1)]: Done  84 tasks      | elapsed:    6.8s
[Paralle

A threshold of 20 was chosen to be 10 times the background rate of ~ 2 photons / pixel, which works well in this case. When the search finishes, a Gaussian is automatically fit to the point cloud and the results are saved to a file (`peaks.nxs`) and printed.

Note that the peak center (`r0`) is not exactly zero, which may indicate a slight error in the diffraction geometry. This error, as well as the extent of the point cloud (`sigma`), should be taken into account when designing a reciprocal space grid for integration. The sampling should not be finer than the geometric errors allow. Here, the Bragg peaks are well localized, and sampling as fine as 20 x 20 x 20 voxels per Bragg peak could be justified (0.5/0.025 = 20). Note that such fine maps consume a lot of computer memory, and thus for the purposes of this tutorial, a coarser map will be used.

Next, create the mask:

In [2]:
!mdx2.mask_peaks geometry.nxs data.nxs peaks.nxs --sigma_cutoff 3

Reading miller_index from geometry.nxs
  importing as MillerIndex from mdx2.geometry
Reading symmetry from geometry.nxs
  importing as Symmetry from mdx2.geometry
Reading image_series from data.nxs
patching virtual field: /entry/image_series/data
  importing as ImageSeries from mdx2.data
Reading peak_model from peaks.nxs
  importing as GaussianPeak from mdx2.geometry
Reading peaks from peaks.nxs
  importing as Peaks from mdx2.data
masking peaks with sigma above threshold: 3.0
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    2.9s
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:    5.1s
[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed:    8.7s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:   12.3s
[Parallel(n_jobs=1)]: Done  24 tasks      | elapsed:   17.4s
[Parallel(n_jobs=1)]: Done  31 tasks      | elapsed:   22.5s
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   29.1s
[Parallel(n_jobs=1)]: Done  49 

The `sigma_cutoff` parameter controlls the size of the masked region around the Bragg peak. A value of 3 means that pixels are masked within 3 standard deviations of the peak center according to the model fit above. The function also masks outliers flagged previously. The resulting masks are saved as an image stack in `mask.nxs` and should be inspected using _NeXpy_ before proceeding.